
### Notebook : 05_prediction_agent


##### 1. Notebook Purpose

The Prediction Agent is responsible for executing churn predictions requested by the Coordinator Agent.

- Reads the Coordinator execution plan.
- Finds the task assigned to prediction_agent.
- Extracts the customer ID from the task.
- Calls the existing Prediction Tool.
- Validates the Prediction Tool response.
- Creates a validated PredictionAgentResult.
- Stores the result in shared state.
- Records execution history and errors.
- Skips cleanly when no prediction task is assigned.

The Prediction Agent does not decide whether a prediction is required. That decision belongs to the Coordinator Agent.


##### 2. Technologies Used

- Python
- Pydantic
- TypedDict
- Callable dependency injection
- Shared multi-agent state
- Databricks notebooks
- Unit testing with mock functions


##### 3. Input

The Prediction Agent receives:

- state: MultiAgentState

- prediction_tool: PredictionToolFunction

The shared state must contain a valid: state["coordinator_result"]


##### 4. Output

The agent returns an updated:

- MultiAgentState

A successful prediction result is stored under: state["agent_results"]["prediction_agent"]


##### 5. Architecture

``` text 

Coordinator Agent
       │
       ▼
CoordinatorResult
       │
       ▼
Prediction Agent
       │
       ├── Find assigned task
       ├── Validate dependencies
       ├── Extract customer ID
       ├── Call Prediction Tool
       ├── Validate tool response
       ├── Create PredictionAgentResult
       └── Store result in shared state
       │
       ▼
Updated MultiAgentState

```


##### 6. Load Shared Models and Helpers

In [0]:
%run ./01_shared_models_code_only

In [0]:
%run ./02_shared_state_and_helpers_code_only


##### 7. Imports

In [0]:
import re

from typing import Any, Callable, Dict, Optional


##### 8. Prediction Tool Contract

In [0]:
PredictionToolFunction = Callable[
    [str],
    Dict[str, Any],
]

# This means the Prediction Tool: Accepts one customer ID as a string. Returns a dictionary.


##### 9. Agent Constants

In [0]:
PREDICTION_AGENT_NAME = "prediction_agent"
SUCCESS_STATUS = "success"
ERROR_STATUS = "error"
SKIPPED_STATUS = "skipped"


##### 10. Find the Prediction Agent Task

In [0]:
# This function searches the Coordinator execution plan for a task assigned to the Prediction Agent.

def find_prediction_agent_task(
    coordinator_result: CoordinatorResult,
) -> Optional[AgentTask]:
    """
    Find the task assigned to the Prediction Agent.

    Parameters
    ----------
    coordinator_result:
        Validated Coordinator Agent result.

    Returns
    -------
    Optional[AgentTask]
        The assigned task, or None when the Prediction Agent
        is not required.
    """

    for task in coordinator_result.execution_plan:
        if task.agent_name == PREDICTION_AGENT_NAME:
            return task

    return None

###### 11. Extract Customer ID

In [0]:
def extract_customer_id_from_text(
    text: str,
) -> Optional[str]:
    """
    Extract a Telco customer ID from text.

    Expected customer ID format:
        XXXX-XXXXX

    Example:
        7590-VHVEG
    """

    if not text:
        return None

    pattern = r"\b[A-Za-z0-9]{4}-[A-Za-z0-9]{5}\b"

    match = re.search(
        pattern,
        text,
        flags=re.IGNORECASE,
    )

    if match is None:
        return None

    return match.group(0).upper()

##### 12. Resolve Customer ID

In [0]:
# This helper first checks the task description and then checks the original user request.

def resolve_customer_id(
    task: AgentTask,
    user_request: str,
) -> str:
    """
    Resolve the customer ID required by the Prediction Tool.

    The task description is checked first because it represents
    the Coordinator's assigned work. The original user request
    is used as a fallback.
    """

    customer_id = extract_customer_id_from_text(
        task.task_description
    )

    if customer_id is None:
        customer_id = extract_customer_id_from_text(
            user_request
        )

    if customer_id is None:
        raise ValueError(
            "Prediction Agent could not determine the customer ID "
            "from the task description or user request."
        )

    return customer_id

##### 13. Convert Prediction Values to Python Types

In [0]:
# Prediction tools may return NumPy values, Spark values, or regular Python values.
# This helper converts common non-Python scalar values into regular Python objects.

def convert_prediction_value(
    value: Any,
) -> Any:
    """
    Convert tool values into standard Python objects.
    """

    if value is None:
        return None

    if hasattr(value, "asDict"):
        return {
            key: convert_prediction_value(item)
            for key, item in value.asDict(
                recursive=True
            ).items()
        }

    if isinstance(value, dict):
        return {
            key: convert_prediction_value(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            convert_prediction_value(item)
            for item in value
        ]

    if isinstance(value, tuple):
        return tuple(
            convert_prediction_value(item)
            for item in value
        )

    if hasattr(value, "item"):
        try:
            return value.item()
        except (ValueError, TypeError):
            pass

    return value


##### 14. Normalize the Prediction Label

In [0]:
# This helper normalizes the result to a Boolean value.

def normalize_prediction(
    prediction: Any,
) -> bool:
    """
    Normalize different prediction formats to a Boolean.

    True:
        Customer is predicted to churn.

    False:
        Customer is predicted not to churn.
    """

    prediction = convert_prediction_value(
        prediction
    )

    if isinstance(prediction, bool):
        return prediction

    if isinstance(prediction, int):
        if prediction in {0, 1}:
            return bool(prediction)

    if isinstance(prediction, float):
        if prediction in {0.0, 1.0}:
            return bool(int(prediction))

    if isinstance(prediction, str):
        normalized = prediction.strip().lower()

        churn_values = {
            "1",
            "true",
            "yes",
            "churn",
            "churned",
            "likely to churn",
        }

        non_churn_values = {
            "0",
            "false",
            "no",
            "not churn",
            "no churn",
            "not likely to churn",
        }

        if normalized in churn_values:
            return True

        if normalized in non_churn_values:
            return False

    raise ValueError(
        f"Unsupported prediction value: {prediction!r}"
    )

##### 15. Normalize Confidence

In [0]:
# The shared Prediction Agent schema expects confidence to be between 0.0 and 1.0.

def normalize_confidence(
    confidence: Any,
) -> float:
    """
    Convert confidence to a float between 0.0 and 1.0.

    Percentage values such as 82 are converted to 0.82.
    """

    confidence = convert_prediction_value(
        confidence
    )

    if confidence is None:
        raise ValueError(
            "Prediction Tool response is missing confidence."
        )

    try:
        normalized_confidence = float(confidence)
    except (TypeError, ValueError) as exc:
        raise ValueError(
            f"Invalid confidence value: {confidence!r}"
        ) from exc

    if 1.0 < normalized_confidence <= 100.0:
        normalized_confidence /= 100.0

    if not 0.0 <= normalized_confidence <= 1.0:
        raise ValueError(
            "Prediction confidence must be between "
            "0.0 and 1.0."
        )

    return normalized_confidence


##### 16. Read Confidence from Different Tool Formats

In [0]:
def get_prediction_confidence(
    tool_response: Dict[str, Any],
) -> float:
    """
    Retrieve confidence from supported Prediction Tool fields.
    """

    supported_fields = [
        "confidence",
        "probability",
        "churn_probability",
    ]

    for field_name in supported_fields:
        if field_name in tool_response:
            return normalize_confidence(
                tool_response[field_name]
            )

    raise ValueError(
        "Prediction Tool response must contain one of: "
        "'confidence', 'probability', or "
        "'churn_probability'."
    )


##### 17. Validate Prediction Tool Response

In [0]:
#This function validates and normalizes the response returned by the injected Prediction Tool. It verifies the execution status, customer ID, prediction value, and confidence before creating a validated PredictionAgentResult.

def validate_prediction_tool_response(
    tool_response: Dict[str, Any],
    expected_customer_id: str,
) -> Dict[str, Any]:
    """
    Validate and normalize the Prediction Tool response.
    """

    if not isinstance(tool_response, dict):
        raise TypeError(
            "Prediction Tool must return a dictionary."
        )

    normalized_response = convert_prediction_value(
        tool_response
    )

    status = normalized_response.get("status")

    if status != SUCCESS_STATUS:
        tool_message = normalized_response.get(
            "message",
            "Prediction Tool execution failed.",
        )

        raise RuntimeError(tool_message)

    returned_customer_id = normalized_response.get(
        "customer_id",
        expected_customer_id,
    )

    returned_customer_id = str(
        returned_customer_id
    ).strip().upper()

    if returned_customer_id != expected_customer_id.upper():
        raise ValueError(
            "Prediction Tool returned a result for a "
            "different customer. "
            f"Expected {expected_customer_id!r}, "
            f"received {returned_customer_id!r}."
        )

    if "prediction" not in normalized_response:
        raise ValueError(
            "Prediction Tool response is missing "
            "'prediction'."
        )

    prediction = normalize_prediction(
        normalized_response["prediction"]
    )

    confidence = get_prediction_confidence(
        normalized_response
    )

    return {
        "customer_id": returned_customer_id,
        "prediction": prediction,
        "confidence": confidence,
        "raw_tool_response": normalized_response,
    }


##### 18. Execute the Prediction Agent

In [0]:
# This function contains the Prediction Agent's core execution logic. It does not modify the shared state directly.

def execute_prediction_agent(
    state: MultiAgentState,
    prediction_tool: PredictionToolFunction,
) -> Optional[PredictionAgentResult]:
    """
    Execute the Prediction Agent's assigned task.

    Returns None when the Coordinator has not assigned
    a prediction task.
    """

    coordinator_result = state.get(
        "coordinator_result"
    )

    if coordinator_result is None:
        raise ValueError(
            "Prediction Agent cannot run because "
            "coordinator_result is missing."
        )

    task = find_prediction_agent_task(
        coordinator_result
    )

    if task is None:
        return None

    validate_task_dependencies(
        state=state,
        task=task,
    )

    user_request = state.get(
        "user_request",
        "",
    )

    customer_id = resolve_customer_id(
        task=task,
        user_request=user_request,
    )

    tool_response = prediction_tool(
        customer_id
    )

    validated_tool_response = (
        validate_prediction_tool_response(
            tool_response=tool_response,
            expected_customer_id=customer_id,
        )
    )

    prediction = validated_tool_response[
        "prediction"
    ]

    confidence = validated_tool_response[
        "confidence"
    ]

    prediction_label = (
        "Churn"
        if prediction
        else "No Churn"
    )

    
    result = PredictionAgentResult(
        agent_name=PREDICTION_AGENT_NAME,
        status=SUCCESS_STATUS,
        message=(  
            f"Customer {customer_id} was predicted as "
            f"{prediction_label} with confidence "
            f"{confidence:.2%}."
        ),
        task_description=task.task_description,
        customer_id=customer_id,
        predicted_category=prediction_label,
        confidence=confidence,
        raw_prediction=prediction,
    )

    return result


##### 19. Run the Prediction Agent

In [0]:
#This function integrates the Prediction Agent with shared state.

def run_prediction_agent(
    state: MultiAgentState,
    prediction_tool: PredictionToolFunction,
) -> MultiAgentState:
    """
    Run the Prediction Agent and update shared state.
    """

    agent_name = PREDICTION_AGENT_NAME

    try:
        result = execute_prediction_agent(
            state=state,
            prediction_tool=prediction_tool,
        )

        if result is None:
            add_execution_history(
                state=state,
                agent_name=agent_name,
                status=SKIPPED_STATUS,
                message=(
                    "Prediction Agent skipped because no "
                    "prediction task was assigned."
                ),
            )

            return state

        store_agent_result(
            state=state,
            agent_name=agent_name,
            result=result,
        )

        add_execution_history(
            state=state,
            agent_name=agent_name,
            status=SUCCESS_STATUS,
            message=result.message,
        )

    except Exception as exc:
        error_message = (
            f"Prediction Agent failed: {exc}"
        )

        add_error(
            state=state,
            agent_name=agent_name,
            error_message=error_message,
        )

        add_execution_history(
            state=state,
            agent_name=agent_name,
            status=ERROR_STATUS,
            message=error_message,
        )

    return state


##### 20. Mock Prediction Tools

In [0]:
# Successful Prediction Tool

def mock_successful_prediction_tool(
    customer_id: str,
) -> Dict[str, Any]:
    """
    Return a predictable successful prediction.
    """

    return {
        "tool": "prediction_tool",
        "status": "success",
        "customer_id": customer_id,
        "prediction": 1,
        "confidence": 0.82,
    }

In [0]:
# Non-Churn Prediction Tool

def mock_non_churn_prediction_tool(
    customer_id: str,
) -> Dict[str, Any]:
    """
    Return a predictable non-churn prediction.
    """

    return {
        "tool": "prediction_tool",
        "status": "success",
        "customer_id": customer_id,
        "prediction": 0,
        "confidence": 0.76,
    }

In [0]:
# Failed Prediction Tool

def mock_failed_prediction_tool(
    customer_id: str,
) -> Dict[str, Any]:
    """
    Simulate a Prediction Tool failure.
    """

    return {
        "tool": "prediction_tool",
        "status": "error",
        "customer_id": customer_id,
        "message": (
            "The prediction endpoint was unavailable."
        ),
    }

In [0]:
# Invalid Prediction Tool

def mock_invalid_prediction_tool(
    customer_id: str,
) -> Dict[str, Any]:
    """
    Simulate a malformed Prediction Tool response.
    """

    return {
        "tool": "prediction_tool",
        "status": "success",
        "customer_id": customer_id,
        "prediction": 1,
        # Confidence intentionally omitted.
    }


##### 21. Test the Prediction Agent

In [0]:
# The test creates a mock Coordinator result rather than running Notebook 03.
#This keeps Notebook 05 focused on unit testing the Prediction Agent.

def test_prediction_agent() -> None:
    """
    Run unit tests for the Prediction Agent.
    """

    print("=" * 80)
    print("TEST 1: Successful churn prediction")
    print("=" * 80)

    state = create_initial_state(
        "Will customer 7590-VHVEG churn?"
    )

    state["coordinator_result"] = CoordinatorResult(
        status=SUCCESS_STATUS,
        message=(
            "Execution plan created successfully."
        ),
        request_type="prediction",
        reasoning=(
            "The user is requesting a churn prediction "
            "for a specific customer."
        ),
        execution_plan=[
            AgentTask(
                task_id="task_1",
                agent_name=PREDICTION_AGENT_NAME,
                task_description=(
                    "Predict churn for customer "
                    "7590-VHVEG."
                ),
                depends_on=[],
            )
        ],
    )

    updated_state = run_prediction_agent(
        state=state,
        prediction_tool=(
            mock_successful_prediction_tool
        ),
    )

    print("Agent results:")
    print(updated_state["agent_results"])

    print("\nErrors:")
    print(updated_state["errors"])

    print("\nExecution history:")
    print(updated_state["execution_history"])

    assert PREDICTION_AGENT_NAME in (
        updated_state["agent_results"]
    )

    result = updated_state[
        "agent_results"
    ][PREDICTION_AGENT_NAME]

    assert result.status == SUCCESS_STATUS
    assert result.predicted_category == "Churn"
    assert result.confidence == 0.82
    assert result.raw_prediction is True
    assert len(updated_state["errors"]) == 0

    print("PASS")
    print(result.model_dump())
    print()


    print("=" * 80)
    print("TEST 2: Successful non-churn prediction")
    print("=" * 80)

    state = create_initial_state(
        "Will customer 5575-GNVDE churn?"
    )

    state["coordinator_result"] = CoordinatorResult(
        status=SUCCESS_STATUS,
        message=(
            "Execution plan created successfully."
        ),
        request_type="prediction",
        reasoning=(
            "The user is requesting a churn prediction "
            "for a specific customer."
        ),
        execution_plan=[
            AgentTask(
                task_id="task_1",
                agent_name=PREDICTION_AGENT_NAME,
                task_description=(
                    "Predict churn for customer "
                    "5575-GNVDE."
                ),
                depends_on=[],
            )
        ],
    )

    updated_state = run_prediction_agent(
        state=state,
        prediction_tool=(
            mock_non_churn_prediction_tool
        ),
    )

    result = updated_state[
        "agent_results"
    ][PREDICTION_AGENT_NAME]

    assert result.status == SUCCESS_STATUS
    assert result.predicted_category == "No Churn"
    assert result.confidence == 0.76
    assert result.raw_prediction is False
    assert len(updated_state["errors"]) == 0

    print("PASS")
    print(result.model_dump())
    print()


    print("=" * 80)
    print("TEST 3: Prediction Agent skips")
    print("=" * 80)

    state = create_initial_state(
        "How many customers churned?"
    )

    state["coordinator_result"] = CoordinatorResult(
        status=SUCCESS_STATUS,
        message=(
            "Execution plan created successfully."
        ),
        request_type="sql_analytics",
        reasoning=(
            "The user is requesting an aggregate "
            "SQL calculation."
        ),
        execution_plan=[
            AgentTask(
                task_id="task_1",
                agent_name="sql_agent",
                task_description=(
                    "Count the number of churned customers."
                ),
                depends_on=[],
            )
        ],
    )

    updated_state = run_prediction_agent(
        state=state,
        prediction_tool=(
            mock_successful_prediction_tool
        ),
    )

    assert PREDICTION_AGENT_NAME not in (
        updated_state["agent_results"]
    )

    assert len(updated_state["errors"]) == 0

    assert (
        updated_state["execution_history"][-1][
            "status"
        ]
        == SKIPPED_STATUS
    )

    print("PASS")
    print(
        updated_state["execution_history"][-1]
    )
    print()


    print("=" * 80)
    print("TEST 4: Prediction Tool failure")
    print("=" * 80)

    state = create_initial_state(
        "Will customer 7590-VHVEG churn?"
    )

    state["coordinator_result"] = CoordinatorResult(
        status=SUCCESS_STATUS,
        message=(
            "Execution plan created successfully."
        ),
        request_type="prediction",
        reasoning=(
            "The user is requesting a churn prediction."
        ),
        execution_plan=[
            AgentTask(
                task_id="task_1",
                agent_name=PREDICTION_AGENT_NAME,
                task_description=(
                    "Predict churn for customer "
                    "7590-VHVEG."
                ),
                depends_on=[],
            )
        ],
    )

    updated_state = run_prediction_agent(
        state=state,
        prediction_tool=(
            mock_failed_prediction_tool
        ),
    )

    assert PREDICTION_AGENT_NAME not in (
        updated_state["agent_results"]
    )

    assert len(updated_state["errors"]) == 1

    assert (
        updated_state["execution_history"][-1][
            "status"
        ]
        == ERROR_STATUS
    )

    print("PASS")
    print(updated_state["errors"][-1])
    print()


    print("=" * 80)
    print("TEST 5: Invalid Prediction Tool response")
    print("=" * 80)

    state = create_initial_state(
        "Will customer 7590-VHVEG churn?"
    )

    state["coordinator_result"] = CoordinatorResult(
        status=SUCCESS_STATUS,
        message=(
            "Execution plan created successfully."
        ),
        request_type="prediction",
        reasoning=(
            "The user is requesting a churn prediction."
        ),
        execution_plan=[
            AgentTask(
                task_id="task_1",
                agent_name=PREDICTION_AGENT_NAME,
                task_description=(
                    "Predict churn for customer "
                    "7590-VHVEG."
                ),
                depends_on=[],
            )
        ],
    )

    updated_state = run_prediction_agent(
        state=state,
        prediction_tool=(
            mock_invalid_prediction_tool
        ),
    )

    assert PREDICTION_AGENT_NAME not in (
        updated_state["agent_results"]
    )

    assert len(updated_state["errors"]) == 1

    assert "confidence" in (
        updated_state["errors"][-1][
            "error_message"
        ].lower()
    )

    print("PASS")
    print(updated_state["errors"][-1])
    print()


    print("=" * 80)
    print("ALL PREDICTION AGENT TESTS PASSED")
    print("=" * 80)

##### 22. Run the Tests

In [0]:
test_prediction_agent()


##### 23. Inspect the Shared Model Fields

In [0]:
#Before running the tests, verify the current schema:
PredictionAgentResult.model_fields.keys()

#You can also inspect all field requirements:
for field_name, field_info in (
    PredictionAgentResult.model_fields.items()
):
    print(
        field_name,
        "required=",
        field_info.is_required(),
    )


##### 24. Connection to the Existing Prediction Tool

In [0]:
state = run_prediction_agent(
    state=state,
    prediction_tool=prediction_tool,
)


##### 25. Key Learnings

###### Specialist agents follow the Coordinator plan

The Prediction Agent does not independently decide whether it should run. It checks the execution plan created by the Coordinator.

###### Shared state connects the agents

The Prediction Agent reads:

state["coordinator_result"]

and writes:

state["agent_results"]["prediction_agent"]

###### Dependency injection reduces coupling

The Prediction Tool is passed into the agent:

run_prediction_agent(
    state=state,
    prediction_tool=prediction_tool,
)

This allows the same Prediction Agent code to work with:

A mock function during unit testing.
The real model endpoint during orchestration.


###### Tool responses require validation

The Prediction Agent should not assume that every tool response is valid. It checks:

- Response type.
- Status.
- Customer ID.
- Prediction value.
- Confidence.
- Confidence range.

###### Unit tests should isolate the agent

Notebook 05 manually creates a mock CoordinatorResult instead of running the real Coordinator Agent.


##### 26. Conclusion

The Prediction Agent now:

- Reads the Coordinator execution plan.
- Finds its assigned task.
- Resolves the customer ID.
- Validates dependencies.
- Calls an injected Prediction Tool.
- Normalizes prediction and confidence values.
- Creates a validated PredictionAgentResult.
- Stores the result in shared state.
- Handles skip and failure scenarios.
- Can be tested independently with mock functions.


###### 27. Next Notebook

The next notebook is : 06_vector_search_agent

The Vector Search Agent will:

- Read the Coordinator execution plan.
- Determine whether semantic search is required.
- Call the existing Vector Search Tool.
- Validate the tool response.
- Create a validated VectorSearchAgentResult.
- Store the result in shared state.
- Skip cleanly when no Vector Search task is assigned.
- Record execution history and errors.